# `54_`にシード数拡大・fold-bagging導入を検証する（`74_`）

## 問題意識

`xxxx.ipynb`（reference著者パイプライン、seed=[42]・StratifiedKFold 2-fold）→
`xxxx_v4.ipynb`（同一パイプライン、7シード・6-fold）で **Public 0.529066 → 0.508039
（-0.021027）** という単一レバーとしては最大級の改善が確認された
（[[modeling-levers-beat-new-features]]の2026-08-18追記）。

一方 `37_`（2026-08-11）では `54_`系アーキテクチャで「シード平均のみ」を単離してPublicで
検証し、**-0.00034（ほぼ無風）**という結果が出ている。この2つは矛盾しない——

| | `54_`系 | `xxxx`系 |
|---|---|---|
| 最終submission生成 | 1回のholdoutでiteration数を決定 → **Train全件×Nシード**で学習 | **StratifiedKFold bagging**（各foldモデルは(K-1)/K件だけで学習）× Nシード |
| foldモデルの学習データ量 | 常に100%（fold概念が無い） | fold数に直結（2-fold=50%、6-fold=83%） |

`54_`は「1モデルあたり全件学習」なのでシードを増やしても分散低減（保険）どまりだが、
`xxxx`は低fold・低シード時に個々のfoldモデルが著しく学習不足かつアンサンブル本数も
極端に少なかった、という仮説。本ノートブックで`54_`側の2つのレバーを**分離して**検証する。

## 事前登録した2つの実験（結果を見てから増減しない）

| 実験 | 変更点 | 変えないもの |
|---|---|---|
| **(a) シード数のみ拡大** | 提出用シードを5→8（`54_`の`SEEDS_VAL`と同一の8本を流用。恣意的選定ではない） | アーキテクチャは`54_`と同一（Train全件学習、fold無し） |
| **(b) fold-bagging導入** | K=6-fold bagging を新規導入（`xxxx_v4`と同じfold数で比較可能に） | シード数は`54_`の提出設定と同じ5本のまま（fold-baggingの効果だけを分離） |

特徴量パイプライン・ハイパーパラメータ（`A_PARAMS`）はすべて`54_`（`R0_memofix_plus_LM`、
444列、Public 0.515030）と同一。[[hyperparameter-retuning-exhausted]]よりハイパーパラメータは
触らない。

## セクション0〜9（セットアップ〜444列特徴量の組み立て）

`73_feature_ideas_on_54.ipynb` と完全に同一のコード（設計上、各ノートブックは自己完結する）。
詳細は割愛し、`74_`固有の実験は第10節以降に置く。

In [1]:
!pip install -q catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 23.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 19.4 MB/s eta 0:00:00


In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "74_seed_fold_scaling_on_54"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")


[2026-08-19 06:04:53] [INFO] === [74_seed_fold_scaling_on_54] 実験開始 ===


INFO:74_seed_fold_scaling_on_54:=== [74_seed_fold_scaling_on_54] 実験開始 ===


[2026-08-19 06:04:53] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260819


INFO:74_seed_fold_scaling_on_54:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260819


[2026-08-19 06:04:53] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/74_seed_fold_scaling_on_54_checkpoint.csv


INFO:74_seed_fold_scaling_on_54:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/74_seed_fold_scaling_on_54_checkpoint.csv


[2026-08-19 06:04:53] [INFO] チェックポイントは未作成（新規実行）


INFO:74_seed_fold_scaling_on_54:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-19 06:04:58] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:74_seed_fold_scaling_on_54:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-19 06:04:58] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:74_seed_fold_scaling_on_54:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-19 06:04:58] [INFO] 定着率: 0.5647


INFO:74_seed_fold_scaling_on_54:定着率: 0.5647


[2026-08-19 06:04:58] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:74_seed_fold_scaling_on_54:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定（改善3の前提）

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、**Test には0名**。

In [7]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。EDA v6の前提が崩れているので調査すること"

[2026-08-19 06:04:59] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:74_seed_fold_scaling_on_54:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-19 06:04:59] [INFO] Test  早期退職者: 0名 / 2502名


INFO:74_seed_fold_scaling_on_54:Test  早期退職者: 0名 / 2502名


[2026-08-19 06:04:59] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:74_seed_fold_scaling_on_54:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-19 06:04:59] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:74_seed_fold_scaling_on_54:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [8]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [9]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-19 06:04:59] [INFO] ------------------------------------------------------------


INFO:74_seed_fold_scaling_on_54:------------------------------------------------------------


[2026-08-19 06:04:59] [INFO] split非依存の基本特徴量を生成中...


INFO:74_seed_fold_scaling_on_54:split非依存の基本特徴量を生成中...


[2026-08-19 06:04:59] [INFO] ------------------------------------------------------------


INFO:74_seed_fold_scaling_on_54:------------------------------------------------------------


[2026-08-19 06:12:57] [INFO] split非依存の基本特徴量生成完了


INFO:74_seed_fold_scaling_on_54:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [10]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-19 06:12:57] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:74_seed_fold_scaling_on_54:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-19 06:12:59] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:74_seed_fold_scaling_on_54:入社時メモ: SVD累積寄与率=0.760


[2026-08-19 06:13:05] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:74_seed_fold_scaling_on_54:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-19 06:13:07] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:74_seed_fold_scaling_on_54:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-19 06:13:07] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:74_seed_fold_scaling_on_54:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [11]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-19 06:13:07] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:74_seed_fold_scaling_on_54:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-19 06:16:07] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:74_seed_fold_scaling_on_54:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [12]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-19 06:16:07] [INFO] Persona単位の基本特徴量を生成中...


INFO:74_seed_fold_scaling_on_54:Persona単位の基本特徴量を生成中...


[2026-08-19 06:16:07] [INFO] Persona単位の基本特徴量処理完了


INFO:74_seed_fold_scaling_on_54:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、49_でパーサーを修正）

`extract_workstyle_section()` に、見出し（`勤務地・働き方：`）が無い書式Bのフォールバックを追加した。
それ以外（`classify_reloc` / `extract_desired_location_v1` / `v2`）は `40_` と同一。


In [13]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m:
        return m.group(1).strip()
    # 49_: 見出しがない書式B（276件、5.24%）のフォールバック。
    # 「勤務地・転居・在宅勤務」に言及する行を拾い、疑似セクションとして返す。
    # 以降のclassify_reloc/extract_desired_location_v1/v2はre.searchで探すだけなので、
    # 複数行を連結してもそのまま動く。
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None


def _report_ws_coverage_fix(train_persona, test_persona):
    """49_の修正がどれだけカバー率を回復させたかをログに残す（診断専用、学習には影響しない）"""
    def old_fn(text):
        if pd.isna(text):
            return None
        m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
        return m.group(1).strip() if m else None

    all_persona = pd.concat([train_persona[["入社時メモ"]], test_persona[["入社時メモ"]]], ignore_index=True)
    ws_old = all_persona["入社時メモ"].apply(old_fn)
    ws_new = all_persona["入社時メモ"].apply(extract_workstyle_section)
    n = len(all_persona)
    logger.info(f"[49_診断] 見出し欠落 修正前 {ws_old.isna().sum()}件({ws_old.isna().sum()/n:.2%}) "
                f"→ 修正後 {ws_new.isna().sum()}件({ws_new.isna().sum()/n:.2%})")


_report_ws_coverage_fix(train_persona, test_persona)


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())


[2026-08-19 06:16:08] [INFO] [49_診断] 見出し欠落 修正前 276件(5.24%) → 修正後 2件(0.04%)


INFO:74_seed_fold_scaling_on_54:[49_診断] 見出し欠落 修正前 276件(5.24%) → 修正後 2件(0.04%)


[2026-08-19 06:16:08] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:74_seed_fold_scaling_on_54:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-19 06:16:08] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:74_seed_fold_scaling_on_54:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-19 06:16:08] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:74_seed_fold_scaling_on_54:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2397
1     364
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2385
1     376
Name: count, dtype: int64


## 5b. L2×M交互作用フラグ（`54_`で追加、Job Embeddedness理論由来）


In [14]:
# ============================================================
# 54_: L2×Mのリスク要因数（Job Embeddedness理論: 複数の埋め込み不足の重なり）
#   ブロックM（専攻職種の分析的ミスマッチ）は29_で単体却下済み（GBDT redundancy）。
#   L2（転居x勤務地_状態_v2 = "非許容_不一致"）との組み合わせをフラグ化する。
#
#   ローカルEDAでの生の効果量（Cochran-Armitage傾向検定, p≈0・機械精度限界）:
#     リスク要因0個(n=1885): 定着率67.0%
#     リスク要因1個(n=653) : 定着率28.8%
#     リスク要因2個(n=47)  : 定着率 2.1%
#   単なるAND(2個該当)だけでなく、0→1→2ときれいな段階的用量反応があったため、
#   二値フラグに加えて順序尺度の"risk_count"も特徴量として渡す。
#   ただし該当セルのp値は採用基準(p<1e-20)を厳密には満たさないセルもあり、
#   単一の事前登録済み検証として扱う（閾値をチューニングしない）。
# ============================================================

_ANALYTICAL_MAJOR = {"情報", "理工学"}
_ANALYTICAL_JOB = {"IT・エンジニアリング", "データ・商品企画・コンサルティング"}


def create_l2_m_interaction_features(persona_df, reloc_v2_df):
    is_analytical_major = persona_df["専攻分野"].isin(_ANALYTICAL_MAJOR)
    is_analytical_job = persona_df["初期職種"].isin(_ANALYTICAL_JOB)
    m_bad = (~is_analytical_major & is_analytical_job).astype(int)

    state = reloc_v2_df.set_index("社員ID").loc[persona_df["社員ID"], "転居x勤務地_状態_v2"].values
    l2_bad = (state == "非許容_不一致").astype(int)

    both_bad = (l2_bad & m_bad)
    risk_count = l2_bad + m_bad  # 0/1/2の順序尺度（用量反応をそのまま渡す）

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "M_不適合": m_bad,
        "L2xM_ダブル不適合": both_bad,
        "L2xM_リスク要因数": risk_count,
    })


train_l2m = create_l2_m_interaction_features(train_persona, train_reloc_v2)
test_l2m = create_l2_m_interaction_features(test_persona, test_reloc_v2)
logger.info(f"L2xMリスク特徴量: Train {train_l2m.shape}, Test {test_l2m.shape}")
print(train_l2m["L2xM_リスク要因数"].value_counts().sort_index())


[2026-08-19 06:16:08] [INFO] L2xMリスク特徴量: Train (2761, 4), Test (2502, 4)


INFO:74_seed_fold_scaling_on_54:L2xMリスク特徴量: Train (2761, 4), Test (2502, 4)


L2xM_リスク要因数
0    2025
1     689
2      47
Name: count, dtype: int64


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"L1"}`/`{"L2"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してL_v1・L_v2のいずれかを単体で追加できるようにする。

In [15]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる。

    28_ からの変更点は2つだけ:
      - split_ratio=1.0 を許容（Train全件学習用。ag_tuningは空になる）
      - exclude_early_from_val=True のとき、検証セットから早期退職者を除く（改善3）
    特徴量の作り方そのものは 28_ と完全に同一。
    '''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        tf = tf.merge(train_l2m, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_l2m, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    # --- 改善3: 検証セットから早期退職者を除く（学習側からは除かない） ---
    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）")

✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）


---
## 9. ベースラインの組み立て（`54_` と同一の444列）

In [16]:
import time                      # 54_ の import セルには入っていないのでここで追加
from sklearn.metrics import log_loss

BLOCK = {"L2"}    # 54_ と同一（L_v2 + L2xM が入る）

logger.info("[全件] 特徴量を組み立て中...")
ag_full, _empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)
assert len(_empty) == 0

def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]

BASE_FEATS = _feature_cols(ag_full)
y_all = ag_full[TARGET_COL].values.astype(int)
print(f"ベースライン {len(BASE_FEATS)} 列 / 学習 {len(ag_full)}名 / Test {len(test_features_full)}名")

# 生存者マスク（採点対象）。Test には早期退職者が0名なので、検証も生存者だけで採点する
IS_SURV = ~ag_full.index.isin(EARLY_LEAVER_IDS)
print(f"生存者 {IS_SURV.sum()}名 / 早期退職者 {(~IS_SURV).sum()}名（学習には使うが採点からは外す）")


[2026-08-19 06:16:08] [INFO] [全件] 特徴量を組み立て中...


INFO:74_seed_fold_scaling_on_54:[全件] 特徴量を組み立て中...


ベースライン 444 列 / 学習 2761名 / Test 2502名
生存者 2632名 / 早期退職者 129名（学習には使うが採点からは外す）


## 10. 実験(a): シード数のみ拡大（5→8、アーキテクチャは`54_`と同一）

`54_`の`SEEDS_SUB=[42,2024,7,1234,99]`（提出用5シード）を、`54_`が診断用に既に使っていた
`SEEDS_VAL`の8シードに差し替えるだけ。Train全件学習・fold無し、というアーキテクチャは変えない。

参考: `54_`の8シードholdout診断val（535名生存者）は既に **0.505477**
（単一シード平均0.508987、sd 0.005181）と分かっている——ここでの新規性はTest予測を
実際に8シードで作りPublicに出すこと。

In [17]:
A_PARAMS = {
    "depth": 4,
    "learning_rate": 0.03518359458951149,
    "l2_leaf_reg": 2.217690447016724,
    "border_count": 218,
    "bagging_temperature": 0.6787467566574921,
    "random_strength": 1.438494697238285,
}
ITER_FULL = 560       # 54_ と同一（変えない）
SEEDS_8 = [42, 2024, 7, 1234, 99, 555, 31337, 2718]   # 54_ の SEEDS_VAL をそのまま流用

obj_cols = [c for c in BASE_FEATS if ag_full[c].dtype == "object"]
X_full = ag_full[BASE_FEATS].fillna(-999)
y_full = ag_full[TARGET_COL].astype(int)
X_test = test_features_full[BASE_FEATS].fillna(-999)

logger.info("-" * 60)
logger.info("実験(a): 8シード Train全件学習を開始")

test_preds_a = []
for seed in SEEDS_8:
    t0 = time.time()
    m = cb.CatBoostClassifier(**A_PARAMS, iterations=ITER_FULL, random_seed=seed,
                               verbose=False, cat_features=obj_cols, task_type="CPU")
    m.fit(X_full, y_full)
    test_preds_a.append(m.predict_proba(X_test)[:, 1])
    logger.info(f"  seed={seed}: 完了 ({time.time()-t0:.0f}秒)")

test_preds_a = np.array(test_preds_a)
pred_5seed_check = test_preds_a[:5].mean(axis=0)   # 最初の5シード = SEEDS_SUBと同一構成
pred_8seed = test_preds_a.mean(axis=0)

# サニティチェック: 最初5シードの平均が既存の54_提出物と近いか(corr)を確認
_ref54 = pd.read_csv(
    sorted((PROJECT_ROOT/"data"/"output").glob("*/*_54_l2_m_interaction_R0_memofix_plus_LM.csv"))[-1],
    header=None, names=[ID_COL, "p"]).set_index(ID_COL).loc[test_ids, "p"].to_numpy()
_corr_check = np.corrcoef(pred_5seed_check, _ref54)[0, 1]
print(f"サニティチェック: 5シード再現版 vs 既存54_提出物の corr = {_corr_check:.4f}"
      f"（CatBoostのセッション間非決定性で完全一致はしない）")
assert _corr_check > 0.98, "54_との対応が大きくズレている。特徴量かパラメータの不一致を疑う"

np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_expA_8seed_testpreds.npy", pred_8seed)
sub_a = pd.DataFrame({ID_COL: test_ids, "定着確率": pred_8seed})
path_a = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_expA_8seed_submission.csv"
sub_a.to_csv(path_a, index=False, header=False)
logger.info(f"実験(a) 提出ファイルを保存: {path_a}")
print(f"\n保存: {path_a}")
print(sub_a.head())

[2026-08-19 06:16:09] [INFO] ------------------------------------------------------------


INFO:74_seed_fold_scaling_on_54:------------------------------------------------------------


[2026-08-19 06:16:09] [INFO] 実験(a): 8シード Train全件学習を開始


INFO:74_seed_fold_scaling_on_54:実験(a): 8シード Train全件学習を開始


[2026-08-19 06:16:16] [INFO]   seed=42: 完了 (7秒)


...

[2026-08-19 06:17:01] [INFO]   seed=2718: 完了 (7秒)


INFO:74_seed_fold_scaling_on_54:  seed=2718: 完了 (7秒)


サニティチェック: 5シード再現版 vs 既存54_提出物の corr = 1.0000（CatBoostのセッション間非決定性で完全一致はしない）
[2026-08-19 06:17:01] [INFO] 実験(a) 提出ファイルを保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_expA_8seed_submission.csv


INFO:74_seed_fold_scaling_on_54:実験(a) 提出ファイルを保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_expA_8seed_submission.csv



保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_expA_8seed_submission.csv
      社員ID      定着確率
0  E000002  0.582238
1  E000003  0.618202
2  E000004  0.802234
3  E000006  0.610756
4  E000009  0.720972


## 11. 実験(b): fold-bagging導入（K=6-fold × 5シード、`54_`と同じ5シードで固定）

各seedごとにStratifiedKFold(6分割)。foldモデルは(K-1)/K≈83%のデータで学習し、
残り1/Kを予測してOOFを埋める。Test予測はK個のfoldモデルの平均。これを5シード分作り、
さらにシード平均する（`xxxx_v4`と同じ二重平均構造）。

反復数は`ITER_HOLDOUT=560`をそのまま流用する（fold学習データ量 83% ≈ `54_`のholdout学習
80%とほぼ同じ規模のため、[[hyperparameter-retuning-exhausted]]の方針通り再チューニングしない）。

採点は生存者(`IS_SURV`)のみ（Testに早期退職者が0名のため、[[test-set-is-survivor-filtered]]）。

In [18]:
from sklearn.model_selection import StratifiedKFold

K_FOLDS = 6                                # xxxx_v4 と揃える
SEEDS_B = [42, 2024, 7, 1234, 99]          # 54_ の提出用シード(SEEDS_SUB)と同一

logger.info("-" * 60)
logger.info(f"実験(b): {K_FOLDS}-fold bagging × {len(SEEDS_B)}シードを開始")

oof_preds_b = np.zeros((len(SEEDS_B), len(X_full)))
test_preds_b = []
for si, seed in enumerate(SEEDS_B):
    t0 = time.time()
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=seed)
    tp_seed = np.zeros(len(X_test))
    for tri, vai in skf.split(X_full, y_full):
        m = cb.CatBoostClassifier(**A_PARAMS, iterations=560, random_seed=seed,
                                   verbose=False, cat_features=obj_cols, task_type="CPU")
        m.fit(X_full.iloc[tri], y_full.iloc[tri])
        oof_preds_b[si, vai] = m.predict_proba(X_full.iloc[vai])[:, 1]
        tp_seed += m.predict_proba(X_test)[:, 1] / K_FOLDS
    test_preds_b.append(tp_seed)
    oof_seed_logloss = log_loss(y_full[IS_SURV], oof_preds_b[si][IS_SURV])
    logger.info(f"  seed={seed}: OOF(生存者) logloss={oof_seed_logloss:.6f} ({time.time()-t0:.0f}秒)")

oof_mean_b = oof_preds_b.mean(axis=0)
test_pred_b = np.mean(test_preds_b, axis=0)
oof_logloss_b = log_loss(y_full[IS_SURV], oof_mean_b[IS_SURV])
print(f"\n実験(b) OOF(生存者{IS_SURV.sum()}名, {len(SEEDS_B)}シード平均) logloss = {oof_logloss_b:.6f}")
print("参考: 54_の8シードholdout診断val(535名, 別の535名split) = 0.505477"
      "（母集団が違うので直接比較にはならない点に注意）")

np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_expB_foldbag_oof.npy", oof_mean_b)
np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_expB_foldbag_testpreds.npy", test_pred_b)
sub_b = pd.DataFrame({ID_COL: test_ids, "定着確率": test_pred_b})
path_b = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_expB_foldbag_submission.csv"
sub_b.to_csv(path_b, index=False, header=False)
logger.info(f"実験(b) 提出ファイルを保存: {path_b}")
print(f"\n保存: {path_b}")
print(sub_b.head())

[2026-08-19 06:17:01] [INFO] ------------------------------------------------------------


INFO:74_seed_fold_scaling_on_54:------------------------------------------------------------


[2026-08-19 06:17:01] [INFO] 実験(b): 6-fold bagging × 5シードを開始


INFO:74_seed_fold_scaling_on_54:実験(b): 6-fold bagging × 5シードを開始


[2026-08-19 06:17:39] [INFO]   seed=42: OOF(生存者) logloss=0.519834 (38秒)


...

[2026-08-19 06:20:09] [INFO]   seed=99: OOF(生存者) logloss=0.517596 (37秒)


INFO:74_seed_fold_scaling_on_54:  seed=99: OOF(生存者) logloss=0.517596 (37秒)



実験(b) OOF(生存者2632名, 5シード平均) logloss = 0.512615
参考: 54_の8シードholdout診断val(535名, 別の535名split) = 0.505477（母集団が違うので直接比較にはならない点に注意）
[2026-08-19 06:20:09] [INFO] 実験(b) 提出ファイルを保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_expB_foldbag_submission.csv


INFO:74_seed_fold_scaling_on_54:実験(b) 提出ファイルを保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_expB_foldbag_submission.csv



保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_expB_foldbag_submission.csv
      社員ID      定着確率
0  E000002  0.629962
1  E000003  0.604078
2  E000004  0.775385
3  E000006  0.602912
4  E000009  0.713083


## 12. まとめ

| 構成 | アーキテクチャ | シード数 | fold数 | Public |
|---|---|---|---|---|
| `54_`（既存最良、比較基準） | Train全件×Nシード | 5 | 無し | **0.515030** |
| 実験(a) | Train全件×Nシード（同一） | **8** | 無し | 要提出確認 |
| 実験(b) | **fold-bagging**（新規） | 5（同一） | **6** | 要提出確認 |

(a)(b)とも単一の事前登録済み変更（[[validation-asymmetry]]の言う「探索による勝者選択ではない」
条件を満たす）。それぞれ1回ずつPublicに提出して確認する。両方を組み合わせた
「8シード×6-fold」（`xxxx_v4`と同じレシピ）は、(a)(b)どちらが効くか切り分けた後の
次の一手として検討する（本NBでは実行しない——一度に複数要因を変えない）。

## 13. xxxx_v4の「在籍月数の回帰モデル」は良いスコアの理由か（分析・2026-08-19追記）

`xxxx_v4.ipynb`は分類器（10年定着ラベル）とは別に、`在籍月数`（0-119か月の実数）を
予測する回帰器も学習し、Platt変換で確率化したうえで分類器と85:15でブレンドしている。
「この回帰情報が良いPublicスコアの理由では」という予想があるので、現時点で分かっている
ことを整理する。

### (1) v1→v4の改善(-0.021027)の説明にはならない

`xxxx.ipynb`(v1, Public 0.529066)と`xxxx_v4.ipynb`(Public 0.508039)の差分を比較すると
（第0節参照）、**変更点はシード数(1→7)とfold数(2→6)だけ**で、回帰ブレンドの仕組み自体は
v1の時点から既に入っていた。つまりこの-0.021027という改善は回帰ブレンドではなく
シード/fold数拡大に起因することが、コード比較から直接わかる
（[[modeling-levers-beat-new-features]]の2026-08-18追記参照）。

### (2) 「回帰単体が分類器単体より良い」という内部数値は要注意

`xxxx_v4`のログには以下の内部val（OOF、2761名全体、較正込み）が残っている:

| 構成 | val logloss |
|---|---|
| 分類器単体（7シード平均） | 0.47875 |
| **回帰→確率 単体** | **0.47361**（分類器単体より良い）|
| 85:15ブレンド | 0.47536 |
| 最終Platt較正後 | 0.47011 |

回帰単体が分類器単体を上回っているように見えるが、`回帰確率_検証`は
**Platt較正器を検証セット自身でin-sample fit&applyしている**
（`Platt変換(検証回帰_平均, 正解ラベル, 検証回帰_平均)`——学習にも適用にも同じ配列を使う）。
これは[[reference-notebook-1st-place-base]]で「`69_`のPublic反転（検証-0.0047改善→
Public+0.0018悪化）の原因として疑われた設計」と**全く同じ楽観バイアスの構造**であり、
この0.47361という数字を額面通り信じるべきではない。

### (3) 現時点で確認できているのはセット全体の効果だけ

「回帰ブレンド込みのxxxx_v4がPublicで良かった」ことは確認済みだが、回帰ブレンドが
その中でどれだけ寄与しているか（分類器単体7シード×6-foldだけでもほぼ同じ数字が出るのか、
回帰ブレンドが実質的な上積みなのか）は**未分離**。クリーンに切り分けるには
「classifier単体 vs 85:15ブレンド」を同一シード/fold数でPublic提出比較する必要があるが、
これは新たな提出枠を要する実験であり本NBのスコープ外（今回は情報を集めるところまで）。

### (4) 今回やったこと: 将来使えるように材料だけ保存

`xxxx_v4.ipynb`第7〜8節に2行だけ追記し、次回実行時に以下を**独立ファイルとして**保存する
ようにした（既存の最終提出ロジックは一切変更していない）:

- `..._xxxx_v4_classifier_only_testpreds.npy`（分類器単体、ブレンド前）
- `..._xxxx_v4_regression_platt_testpreds.npy`（回帰由来Platt確率、ブレンド前）

次節のアンサンブルは、in-sampleバイアスのある内部val数値を信じるのではなく、
**ラベルを使わないTest予測レベルの相関・多様性**に基づいて回帰由来の確率を評価する
（見つかれば自動的に7つ目の材料として組み込む。まだ`xxxx_v4`を再実行していないので
現時点ではファイルが存在せず、6モデル構成のまま進む）。

## 14. 最終アンサンブル（54_(a) / 54_(b) / xxxx_v4 / TabPFN(hire_fixed) / TabICL(hire_fixed) / TabPFN(top150)）

AutoGluonを使わない6つの材料を単純平均する。[[ensemble-oof-overfitting]]の教訓通り
重み探索はしない（非チューニングの等重み平均が既定）。54_(a)・54_(b)は本NB第10・11節の
結果をそのまま使い、残り4つは保存済みファイルから読み込む。

In [19]:
def _find(pat):
    h = sorted((PROJECT_ROOT / "data" / "output").glob(pat))
    return h[-1] if h else None


def _load_csv_test(pat):
    p = _find(pat)
    assert p is not None, f"見つからない: {pat}（先に該当ノートブックを実行すること）"
    v = pd.read_csv(p, header=None, names=[ID_COL, "p"]).set_index(ID_COL).loc[test_ids, "p"].to_numpy()
    return v, p.name


def _load_npy_test(pat):
    p = _find(pat)
    assert p is not None, f"見つからない: {pat}（先に該当ノートブックを実行すること）"
    v = np.load(p)
    assert len(v) == len(test_ids), f"{p.name}: 行数が一致しない"
    return v, p.name


components = {}
components["54_(a)8seed"] = pred_8seed        # 第10節で計算済み
components["54_(b)foldbag"] = test_pred_b      # 第11節で計算済み

for label, pat, kind in [
    ("xxxx_v4",           "*/*xxxx_v4_submission.csv",                                                     "csv"),
    ("TabPFN(hire_fixed)", "*/*_70_hire_fixed_fm_submission_tabpfn_hire_fixed_testpreds.npy",               "npy"),
    ("TabICL(hire_fixed)", "*/*_70_hire_fixed_fm_submission_tabicl_hire_fixed_testpreds.npy",               "npy"),
    ("TabPFN(top150)",     "*/*_63_tabpfn_ensemble_tabpfn_top150_testpreds.npy",                            "npy"),
]:
    v, fname = (_load_csv_test(pat) if kind == "csv" else _load_npy_test(pat))
    components[label] = v
    msg = f"【再利用】{label}: {fname} — 本NBでは学習していない"
    print(msg); logger.info(msg)

# 回帰由来Platt確率(第13節)。xxxx_v4を再実行済みなら自動で7つ目として追加、無ければスキップ
p_reg = _find("*/*xxxx_v4_regression_platt_testpreds.npy")
if p_reg is not None:
    v_reg = np.load(p_reg)
    assert len(v_reg) == len(test_ids)
    components["xxxx_v4_回帰Platt(参考)"] = v_reg
    print(f"【追加】xxxx_v4の回帰由来Platt確率が見つかったため7つ目として追加: {p_reg.name}")
    logger.info(f"回帰由来Platt確率を追加: {p_reg.name}")
else:
    print("xxxx_v4の回帰由来Platt確率はまだ無い（xxxx_v4.ipynb再実行後に生成される）。6モデル構成のまま進める。")

names = list(components)
print(f"\n=== 相関行列（参考、ノイズ床0.02122） ===")
print(f'{"":22s}' + "".join(f"{n[:14]:>16s}" for n in names))
for a in names:
    row = f"{a:22s}"
    for b in names:
        row += f"{np.corrcoef(components[a], components[b])[0,1]:16.4f}"
    print(row)

ensemble_mean = np.mean([components[n] for n in names], axis=0)

sub = pd.DataFrame({ID_COL: test_ids, "定着確率": ensemble_mean})
n_models = len(names)
path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_final{n_models}_ensemble_mean.csv"
sub.to_csv(path, index=False, header=False)
logger.info(f"最終アンサンブル({n_models}モデル等重み平均)を保存: {path}")
print(f"\n保存: {path}")
print(f"構成({n_models}モデル等重み平均): {names}")
print(sub.head())

【再利用】xxxx_v4: 20260817_xxxx_v4_submission.csv — 本NBでは学習していない
[2026-08-19 06:20:09] [INFO] 【再利用】xxxx_v4: 20260817_xxxx_v4_submission.csv — 本NBでは学習していない


INFO:74_seed_fold_scaling_on_54:【再利用】xxxx_v4: 20260817_xxxx_v4_submission.csv — 本NBでは学習していない


【再利用】TabPFN(hire_fixed): 20260816_70_hire_fixed_fm_submission_tabpfn_hire_fixed_testpreds.npy — 本NBでは学習していない
[2026-08-19 06:20:09] [INFO] 【再利用】TabPFN(hire_fixed): 20260816_70_hire_fixed_fm_submission_tabpfn_hire_fixed_testpreds.npy — 本NBでは学習していない


INFO:74_seed_fold_scaling_on_54:【再利用】TabPFN(hire_fixed): 20260816_70_hire_fixed_fm_submission_tabpfn_hire_fixed_testpreds.npy — 本NBでは学習していない


【再利用】TabICL(hire_fixed): 20260816_70_hire_fixed_fm_submission_tabicl_hire_fixed_testpreds.npy — 本NBでは学習していない
[2026-08-19 06:20:09] [INFO] 【再利用】TabICL(hire_fixed): 20260816_70_hire_fixed_fm_submission_tabicl_hire_fixed_testpreds.npy — 本NBでは学習していない


...

【再利用】TabPFN(top150): 20260816_63_tabpfn_ensemble_tabpfn_top150_testpreds.npy — 本NBでは学習していない
[2026-08-19 06:20:10] [INFO] 【再利用】TabPFN(top150): 20260816_63_tabpfn_ensemble_tabpfn_top150_testpreds.npy — 本NBでは学習していない


INFO:74_seed_fold_scaling_on_54:【再利用】TabPFN(top150): 20260816_63_tabpfn_ensemble_tabpfn_top150_testpreds.npy — 本NBでは学習していない


xxxx_v4の回帰由来Platt確率はまだ無い（xxxx_v4.ipynb再実行後に生成される）。6モデル構成のまま進める。

=== 相関行列（参考、ノイズ床0.02122） ===
                           54_(a)8seed   54_(b)foldbag         xxxx_v4  TabPFN(hire_fi  TabICL(hire_fi  TabPFN(top150)
54_(a)8seed                     1.0000          0.9960          0.8905          0.9138          0.9194          0.9572
54_(b)foldbag                   0.9960          1.0000          0.8878          0.9124          0.9264          0.9611
xxxx_v4                         0.8905          0.8878          1.0000          0.8843          0.8693          0.9011
TabPFN(hire_fixed)              0.9138          0.9124          0.8843          1.0000          0.9421          0.9571
TabICL(hire_fixed)              0.9194          0.9264          0.8693          0.9421          1.0000          0.9524
TabPFN(top150)                  0.9572          0.9611          0.9011          0.9571          0.9524          1.0000
[2026-08-19 06:20:10] [INFO] 最終アンサンブル(6モデル等重み平均)を保存: /content/drive/MyDri

INFO:74_seed_fold_scaling_on_54:最終アンサンブル(6モデル等重み平均)を保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_final6_ensemble_mean.csv



保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_final6_ensemble_mean.csv
構成(6モデル等重み平均): ['54_(a)8seed', '54_(b)foldbag', 'xxxx_v4', 'TabPFN(hire_fixed)', 'TabICL(hire_fixed)', 'TabPFN(top150)']
      社員ID      定着確率
0  E000002  0.626210
1  E000003  0.578003
2  E000004  0.751972
3  E000006  0.554479
4  E000009  0.730157


## 15. 回帰ブレンド実験の準備（在籍月数ラベル・Platt関数、2026-08-19追記）

`69_catboost_native_text.ipynb`が移植した「在籍月数回帰+Platt較正+85:15ブレンド」は
`54_`アーキテクチャで一度Public不採用が確定している（検証-0.0047→Public+0.001831悪化）。
ただし`69_`は**検証(holdout)は8シード平均、実際の提出(Test)は5シード平均**という
食い違いがあった。以下2つを切り分けて再検証する:

- **実験(c)**: 実験(b)のfold-bagging分類器（K=6×5シード）に、同じfold-baggingで
  作った回帰OOFをブレンドする（fold-baggingアーキテクチャ×回帰ブレンドの組み合わせ）
- **実験(d)**: `69_`と全く同じ全件学習アーキテクチャのまま、val/Testとも8シードに統一する
  （`69_`の val=8シード/Test=5シードという食い違いを解消しただけの構成）

In [20]:
from sklearn.linear_model import LogisticRegression

train_monthly_full = pd.read_csv(INPUT_DIR / "employee_monthly_train_full.csv")
TENURE = train_monthly_full.groupby(ID_COL)["経過月数"].max()
logger.info(f"在籍月数(TENURE): 平均={TENURE.mean():.1f} 最小={TENURE.min():.0f} 最大={TENURE.max():.0f}")

y_tenure_full = TENURE.reindex(ag_full.index).to_numpy()
assert not np.isnan(y_tenure_full).any(), "TENUREにマッチしない社員IDがある"

MIX_RATIO = 0.85   # reference/69_と同一の固定値。ここでは走査しない


def _to_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def _platt_fit(x, y):
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(x).reshape(-1, 1), y)
    return lr


def _platt_apply(lr, x):
    return lr.predict_proba(np.asarray(x).reshape(-1, 1))[:, 1]


print("✅ TENURE・Platt関数の準備完了")

[2026-08-19 08:46:46] [INFO] 在籍月数(TENURE): 平均=92.3 最小=11 最大=119


INFO:74_seed_fold_scaling_on_54:在籍月数(TENURE): 平均=92.3 最小=11 最大=119


✅ TENURE・Platt関数の準備完了


## 16. 実験(c): fold-bagging分類器（実験b）に回帰ブレンドを追加

実験(b)の`oof_mean_b`/`test_pred_b`（K=6×5シード）を分類器側として再利用し、
**同じfold構造・同じシードで回帰器も学習**してPlatt→85:15ブレンド→最終較正まで通す。
最終較正は`xxxx_v4`と同じ設計（生存者部分集合でネストKFold fit、全体に適用）を使う。

In [21]:
K_FOLDS_C = 6
SEEDS_C = [42, 2024, 7, 1234, 99]   # 実験(b)と同一シード（分類器側と揃える）

logger.info("-" * 60)
logger.info(f"実験(c): fold-bagging({K_FOLDS_C}-fold x {len(SEEDS_C)}シード)の回帰ブレンドを開始")

oof_reg_c = np.zeros((len(SEEDS_C), len(X_full)))
test_reg_c = []
for si, seed in enumerate(SEEDS_C):
    t0 = time.time()
    skf = StratifiedKFold(n_splits=K_FOLDS_C, shuffle=True, random_state=seed)
    tp_seed = np.zeros(len(X_test))
    for tri, vai in skf.split(X_full, y_full):   # y_fullで層化(分類器=実験bと同じ分割)
        m = cb.CatBoostRegressor(**A_PARAMS, loss_function="RMSE", eval_metric="RMSE",
                                  iterations=560, random_seed=seed,
                                  verbose=False, cat_features=obj_cols, task_type="CPU")
        m.fit(X_full.iloc[tri], y_tenure_full[tri])
        oof_reg_c[si, vai] = m.predict(X_full.iloc[vai])
        tp_seed += m.predict(X_test) / K_FOLDS_C
    test_reg_c.append(tp_seed)
    logger.info(f"  [回帰] seed={seed}: 完了 ({time.time()-t0:.0f}秒)")

reg_oof_c = oof_reg_c.mean(axis=0)
reg_test_c = np.mean(test_reg_c, axis=0)

reg_platt_c = _platt_fit(reg_oof_c, y_full)
reg_prob_oof_c = _platt_apply(reg_platt_c, reg_oof_c)
reg_prob_test_c = _platt_apply(reg_platt_c, reg_test_c)

blend_oof_c = MIX_RATIO * oof_mean_b + (1 - MIX_RATIO) * reg_prob_oof_c
blend_test_c = MIX_RATIO * test_pred_b + (1 - MIX_RATIO) * reg_prob_test_c

z_oof_c = _to_logit(blend_oof_c)
calibrated_oof_c = np.zeros(len(z_oof_c))
for tri, vai in KFold(n_splits=5, shuffle=True, random_state=27).split(z_oof_c):
    tri_surv = tri[IS_SURV[tri]]
    cal = _platt_fit(z_oof_c[tri_surv], y_full[tri_surv])
    calibrated_oof_c[vai] = _platt_apply(cal, z_oof_c[vai])

final_cal_c = _platt_fit(z_oof_c[IS_SURV], y_full[IS_SURV])
calibrated_test_c = _platt_apply(final_cal_c, _to_logit(blend_test_c))

print(f"参考: 実験(c)の分類器単体(=実験b, 生存者)  logloss = {log_loss(y_full[IS_SURV], oof_mean_b[IS_SURV]):.6f}")
print(f"実験(c) 回帰→確率単体(生存者)              logloss = {log_loss(y_full[IS_SURV], reg_prob_oof_c[IS_SURV]):.6f}")
print(f"実験(c) ブレンド後(85:15, 生存者)          logloss = {log_loss(y_full[IS_SURV], blend_oof_c[IS_SURV]):.6f}")
print(f"実験(c) 最終Platt較正後(生存者)             logloss = {log_loss(y_full[IS_SURV], calibrated_oof_c[IS_SURV]):.6f}")

np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_expC_foldbag_blend_testpreds.npy", calibrated_test_c)
sub_c = pd.DataFrame({ID_COL: test_ids, "定着確率": calibrated_test_c})
path_c = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_expC_foldbag_blend_submission.csv"
sub_c.to_csv(path_c, index=False, header=False)
logger.info(f"実験(c) 提出ファイルを保存: {path_c}")
print(f"\n保存: {path_c}")
print(sub_c.head())

[2026-08-19 08:46:48] [INFO] ------------------------------------------------------------


INFO:74_seed_fold_scaling_on_54:------------------------------------------------------------


[2026-08-19 08:46:48] [INFO] 実験(c): fold-bagging(6-fold x 5シード)の回帰ブレンドを開始


INFO:74_seed_fold_scaling_on_54:実験(c): fold-bagging(6-fold x 5シード)の回帰ブレンドを開始


[2026-08-19 08:47:19] [INFO]   [回帰] seed=42: 完了 (31秒)


...

[2026-08-19 08:49:25] [INFO]   [回帰] seed=99: 完了 (32秒)


INFO:74_seed_fold_scaling_on_54:  [回帰] seed=99: 完了 (32秒)


参考: 実験(c)の分類器単体(=実験b, 生存者)  logloss = 0.512615
実験(c) 回帰→確率単体(生存者)              logloss = 0.508796
実験(c) ブレンド後(85:15, 生存者)          logloss = 0.510130
実験(c) 最終Platt較正後(生存者)             logloss = 0.509681
[2026-08-19 08:49:25] [INFO] 実験(c) 提出ファイルを保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_expC_foldbag_blend_submission.csv


INFO:74_seed_fold_scaling_on_54:実験(c) 提出ファイルを保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_expC_foldbag_blend_submission.csv



保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_expC_foldbag_blend_submission.csv
      社員ID      定着確率
0  E000002  0.638254
1  E000003  0.616026
2  E000004  0.790336
3  E000006  0.599822
4  E000009  0.724299


## 17. 実験(d): `69_`の構成をval/Testとも8シードに統一

`69_`は全件学習アーキテクチャ（fold無し）で、検証(holdout)は`SEEDS_VAL`8シード、
実際のTest予測は`SEEDS_SUB`5シードという食い違いがあった。ここではval/Testとも
`SEEDS_8`（実験(a)と同じ8シード）に統一し、`69_`と同じ80%ホールドアウト分割
（`prepare_split`は決定的なので同一分割が再現される）でPlatt較正器をfitする。
分類器のTest予測は実験(a)の`pred_8seed`をそのまま再利用する。

In [22]:
ITER_HOLDOUT = 560   # 54_/69_ と同一

logger.info("-" * 60)
logger.info("実験(d): 69_の構成(全件学習+ホールドアウトPlatt較正)をシード8本に統一して再実行")

ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)
print(f"80%ホールドアウト: 学習{len(ag_train_80b)}名 / 検証(生存者){len(ag_val_surv)}名")

obj_cols_80 = [c for c in BASE_FEATS if ag_train_80b[c].dtype == "object"]
X_tr80 = ag_train_80b[BASE_FEATS].fillna(-999)
y_tr80 = ag_train_80b[TARGET_COL].astype(int)
X_va = ag_val_surv[BASE_FEATS].fillna(-999)
y_va = ag_val_surv[TARGET_COL].astype(int).to_numpy()
y_tenure_tr80 = TENURE.reindex(ag_train_80b.index).to_numpy()
assert not np.isnan(y_tenure_tr80).any(), "TENUREにマッチしない社員IDがある(80%学習側)"


def _fit_avg_classifier(X_tr, y_tr, X_pred, seeds):
    preds = []
    for seed in seeds:
        m = cb.CatBoostClassifier(**A_PARAMS, iterations=ITER_HOLDOUT, random_seed=seed,
                                   verbose=False, cat_features=obj_cols_80, task_type="CPU")
        m.fit(X_tr, y_tr)
        preds.append(m.predict_proba(X_pred)[:, 1])
    return np.mean(preds, axis=0)


def _fit_avg_regressor(X_tr, y_tr, X_pred, seeds):
    preds = []
    for seed in seeds:
        m = cb.CatBoostRegressor(**A_PARAMS, loss_function="RMSE", eval_metric="RMSE",
                                  iterations=ITER_HOLDOUT, random_seed=seed,
                                  verbose=False, cat_features=obj_cols_80, task_type="CPU")
        m.fit(X_tr, y_tr)
        preds.append(m.predict(X_pred))
    return np.mean(preds, axis=0)


t0 = time.time()
cls_val_d = _fit_avg_classifier(X_tr80, y_tr80, X_va, SEEDS_8)
logger.info(f"  分類器(holdout, 8シード) 完了 ({time.time()-t0:.0f}秒)")

t0 = time.time()
reg_val_d = _fit_avg_regressor(X_tr80, y_tenure_tr80, X_va, SEEDS_8)
logger.info(f"  回帰器(holdout, 8シード) 完了 ({time.time()-t0:.0f}秒)")

t0 = time.time()
reg_test_d = _fit_avg_regressor(X_full, y_tenure_full, X_test, SEEDS_8)
logger.info(f"  回帰器(全件学習, 8シード) 完了 ({time.time()-t0:.0f}秒)")

cls_test_d = pred_8seed   # 実験(a)で計算済み(8シード全件学習)を再利用

reg_platt_d = _platt_fit(reg_val_d, y_va)
reg_prob_val_d = _platt_apply(reg_platt_d, reg_val_d)
reg_prob_test_d = _platt_apply(reg_platt_d, reg_test_d)

blend_val_d = MIX_RATIO * cls_val_d + (1 - MIX_RATIO) * reg_prob_val_d
blend_test_d = MIX_RATIO * cls_test_d + (1 - MIX_RATIO) * reg_prob_test_d

z_val_d = _to_logit(blend_val_d)
calibrated_val_d = np.zeros(len(z_val_d))
for tri, vai in KFold(n_splits=5, shuffle=True, random_state=27).split(z_val_d):
    cal = _platt_fit(z_val_d[tri], y_va[tri])
    calibrated_val_d[vai] = _platt_apply(cal, z_val_d[vai])

final_cal_d = _platt_fit(z_val_d, y_va)
calibrated_test_d = _platt_apply(final_cal_d, _to_logit(blend_test_d))

print(f"\n実験(d) 分類器単体(holdout, 8シード)  logloss = {log_loss(y_va, cls_val_d):.6f}")
print(f"実験(d) 回帰→確率単体                 logloss = {log_loss(y_va, reg_prob_val_d):.6f}")
print(f"実験(d) ブレンド後(85:15)             logloss = {log_loss(y_va, blend_val_d):.6f}")
print(f"実験(d) 最終Platt較正後(OOF)          logloss = {log_loss(y_va, calibrated_val_d):.6f}")
print("参考(69_の元の結果, val=8シード/Test=5シードの食い違いあり): "
      "検証-0.004677→Public+0.001831悪化")

np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_expD_8seed_blend_testpreds.npy", calibrated_test_d)
sub_d = pd.DataFrame({ID_COL: test_ids, "定着確率": calibrated_test_d})
path_d = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_expD_8seed_blend_submission.csv"
sub_d.to_csv(path_d, index=False, header=False)
logger.info(f"実験(d) 提出ファイルを保存: {path_d}")
print(f"\n保存: {path_d}")
print(sub_d.head())

[2026-08-19 08:49:31] [INFO] ------------------------------------------------------------


INFO:74_seed_fold_scaling_on_54:------------------------------------------------------------


[2026-08-19 08:49:31] [INFO] 実験(d): 69_の構成(全件学習+ホールドアウトPlatt較正)をシード8本に統一して再実行


INFO:74_seed_fold_scaling_on_54:実験(d): 69_の構成(全件学習+ホールドアウトPlatt較正)をシード8本に統一して再実行


[2026-08-19 08:49:32] [INFO]   検証セット: 553 → 535件（早期退職者18名を除外）


...

[2026-08-19 08:51:47] [INFO]   回帰器(全件学習, 8シード) 完了 (44秒)


INFO:74_seed_fold_scaling_on_54:  回帰器(全件学習, 8シード) 完了 (44秒)



実験(d) 分類器単体(holdout, 8シード)  logloss = 0.505477
実験(d) 回帰→確率単体                 logloss = 0.509100
実験(d) ブレンド後(85:15)             logloss = 0.503394
実験(d) 最終Platt較正後(OOF)          logloss = 0.500800
参考(69_の元の結果, val=8シード/Test=5シードの食い違いあり): 検証-0.004677→Public+0.001831悪化
[2026-08-19 08:51:47] [INFO] 実験(d) 提出ファイルを保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_expD_8seed_blend_submission.csv


INFO:74_seed_fold_scaling_on_54:実験(d) 提出ファイルを保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_expD_8seed_blend_submission.csv



保存: /content/drive/MyDrive/jaggle_2026/data/output/20260819/20260819_74_seed_fold_scaling_on_54_expD_8seed_blend_submission.csv
      社員ID      定着確率
0  E000002  0.537673
1  E000003  0.576990
2  E000004  0.789490
3  E000006  0.552324
4  E000009  0.691571


## 18. まとめ（回帰ブレンド実験、2026-08-19追記）

| 構成 | アーキテクチャ | 回帰ブレンド | Public |
|---|---|---|---|
| `54_`（比較基準） | 全件学習×5シード | なし | 0.515030 |
| 実験(a) | 全件学習×8シード | なし | 0.515714（無風） |
| 実験(b) | fold-bagging(6×5) | なし | 0.518509（やや悪化） |
| `69_`（過去の失敗） | 全件学習（val8シード/Test5シード） | あり | 0.517303（+0.0018悪化） |
| **実験(c)** | fold-bagging(6×5) | **あり** | 要提出確認 |
| **実験(d)** | 全件学習×8シード（val/Test統一） | **あり** | 要提出確認 |
| 参考: `xxxx_v4`分類器単体 | reference特徴量, 7×6 | なし | 0.512683 |
| 参考: `xxxx_v4`（実際の提出物） | reference特徴量, 7×6 | あり | 0.508039 |

(c)(d)とも単一の事前登録済み変更。(c)は「fold-baggingが回帰ブレンドを機能させる土台になるか」、
(d)は「val/Testのシード数を揃えるだけで`69_`の反転が消えるか」を、それぞれ切り分けて検証する。